In [3]:
#importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/train.csv')
irrigation.head()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier # Changed to Classifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression # Changed to Classifier

# 1. Separate your features (X) and target (y) FIRST
X = irrigation.drop(['id', 'Irrigation_Need'], axis=1)
y = irrigation['Irrigation_Need'] # Target remains a single column!

# 2. Categorical encoding ONLY on features (X)
X_encoded = pd.get_dummies(X, drop_first=True).astype(int)

# 3. Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 4. Scaling the features (for Logistic Regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline Logistic Regression Model
baseline_lr = LogisticRegression(max_iter=1000)
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)

# Baseline Random Forest Classifier
baseline_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf) 

# Print baseline results
print(f'Baseline Logistic Regression Accuracy: {acc_lr:.4f}')
print(f'Baseline Random Forest Accuracy: {acc_rf:.4f}')


Baseline Logistic Regression Accuracy: 0.8751
Baseline Random Forest Accuracy: 0.9850


In [8]:
# CV with baseline models using accuracy as the scoring metric
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='accuracy')
print(f'Baseline Logistic Regression CV Accuracy Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV Accuracy Scores: {cv_scores_rf}')


Baseline Logistic Regression CV Accuracy Scores: [0.87753968 0.87646825 0.87627976 0.87831349 0.87522817]
Baseline Random Forest CV Accuracy Scores: [0.98536706 0.98460317 0.98507937 0.98411706 0.9841369 ]


In [10]:
# gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    'n_estimators': [100],             # Dropped 200 to save time
    'max_depth': [10, 20],             # Removed 'None' because unbounded trees take forever
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']           # 'sqrt' is standard, dropped 'log2' to save time
}           

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)  # Changed cv=5 to cv=3
grid_search_rf.fit(X_train, y_train)    

print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV Accuracy Score: {grid_search_rf.best_score_:.4f}')


Best RF Hyperparameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
Best RF CV Accuracy Score: 0.9846


In [11]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
   n_estimators  max_depth  min_samples_split max_features  mean_test_score
2           100         20                  2         sqrt         0.984621
3           100         20                  5         sqrt         0.984548
0           100         10                  2         sqrt         0.972833
1           100         10                  5         sqrt         0.970748


In [12]:
import pandas as pd

# 1. Load the exact test dataset
test_data = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/test.csv')

# Save test IDs and drop the ID column
test_ids = test_data['id']
X_test_raw = test_data.drop(['id'], axis=1) 

# 2. Dummy encode the test data (just like you did in the beginning of this notebook!) 
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True).astype(int)

# Crucial: This perfectly aligns your test set columns with your X_train columns, 
# filling any missing dummy features with 0 to prevent crashes
X_test_encoded = X_test_encoded.reindex(columns=X_train.columns, fill_value=0)

# 3. Predict using your Random Forest! 
# (Note: If you used GridSearchCV, change 'baseline_rf' to 'grid_search_rf')
rf_predictions = baseline_rf.predict(X_test_encoded)

# 4. Create your submission DataFrame (no mapping needed because RF output words directly)
submission_rf = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': rf_predictions
})

# 5. Save to your Homework-2 folder without index!
submission_rf.to_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/random_forest_submission.csv', index=False)

print("Your file is ready! Use Kaggle to submit 'random_forest_submission.csv'")


Your file is ready! Use Kaggle to submit 'random_forest_submission.csv'
